# Plan-and-Execute

이번 강의에서는 복잡한 멀티스텝 태스크를 처리하는 Plan-and-Execute 패턴을 배운다. 전체 계획을 먼저 세우고, 단계별로 실행하며, 중간 결과에 따라 계획을 수정(Re-planning)하는 구조다.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Plan-and-Execute란

ReAct Agent는 **한 번에 한 단계씩** 생각하고 행동한다. 간단한 질문에는 잘 동작하지만, 복잡한 태스크에서는 한계가 있다.

| | ReAct | Plan-and-Execute |
|---|---|---|
| 전략 | 한 단계씩 결정 | 전체 계획을 먼저 수립 |
| 장점 | 단순, 빠름 | 복잡한 태스크에 강함 |
| 단점 | 큰 그림을 놓칠 수 있음 | 계획 수립에 비용 발생 |
| 적합한 경우 | 단일 검색, 간단한 질의 | 멀티스텝 조사, 보고서 작성 |

예를 들어 "경쟁사 3곳의 AI 서비스 가격을 조사해서 비교표를 만들고 보고서를 작성해줘"라는 요청을 생각해보자.

- **ReAct**: 첫 번째 경쟁사를 검색 -> 결과를 보고 다음을 결정 -> ... (방향을 잃기 쉬움)
- **Plan-and-Execute**: 먼저 "1) A사 가격 조사 2) B사 가격 조사 3) C사 가격 조사 4) 비교표 작성 5) 보고서 작성"이라는 계획을 세우고, 하나씩 실행

핵심 흐름은 다음과 같다.

```
입력 → Planner(계획 수립) → Executor(단계 실행) → Replanner(계획 수정) → ... → 최종 응답
```

## State 설계

Plan-and-Execute의 State는 이전보다 복잡하다. 계획, 실행 결과, 최종 응답을 모두 추적해야 한다.

| 필드 | 타입 | 역할 |
|------|------|------|
| `input` | `str` | 사용자의 원래 요청 |
| `plan` | `list[str]` | 현재 계획 (단계 목록) |
| `past_steps` | `list[tuple]` | 완료된 단계와 결과 (누적) |
| `response` | `str` | 최종 응답 |

`past_steps`에는 `operator.add` reducer를 사용해서 실행 결과가 계속 누적되도록 한다. `plan`은 Replanner가 통째로 교체할 수 있어야 하므로 reducer 없이 덮어쓰기로 둔다.

State 설계 판단 기준:
- **누적이 필요한 데이터** (실행 이력) -> reducer 사용
- **최신 값만 필요한 데이터** (현재 계획, 최종 응답) -> reducer 없이 덮어쓰기
- **불필요한 데이터는 넣지 않는다** -> State가 커지면 LLM에 전달할 컨텍스트도 커져서 비용이 증가한다

In [ ]:
import operator
from typing import Annotated, TypedDict


class PlanExecuteState(TypedDict):
    input: str
    plan: list[str]
    past_steps: Annotated[list[tuple], operator.add]
    response: str

## Tool 정의

Executor가 사용할 Tool을 먼저 정의한다. 여기서는 웹 검색 도구를 사용한다.

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

search = TavilySearchResults(max_results=3)

tools = [search]

## 노드 구현

3개의 핵심 노드를 만든다.

1. **Planner** - 사용자 입력을 받아 실행 계획을 생성
2. **Executor** - 계획의 현재 단계를 실행
3. **Replanner** - 중간 결과를 보고 계획을 수정하거나 최종 응답을 생성

### Planner

Structured Output으로 계획을 리스트 형태로 받는다.

In [ ]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


class Plan(BaseModel):
    """실행 계획"""
    steps: list[str] = Field(description="수행해야 할 단계들의 목록 (순서대로)")


planner_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """당신은 주어진 목표를 달성하기 위한 단계별 계획을 세우는 전문가입니다.
각 단계는 구체적이고 실행 가능해야 합니다.
불필요한 단계는 포함하지 마세요. 최종 단계의 결과가 곧 최종 응답이 됩니다.""",
    ),
    ("human", "{input}"),
])

planner_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
planner = planner_prompt | planner_llm.with_structured_output(Plan)

In [ ]:
# Planner 단독 테스트
result = planner.invoke({"input": "경쟁사 AI 서비스 가격을 조사해서 비교 보고서를 만들어줘"})
print("생성된 계획:")
for i, step in enumerate(result.steps, 1):
    print(f"  {i}. {step}")

### Executor

현재 단계를 실행하는 Agent다. Tool을 사용할 수 있어야 하므로 ReAct 스타일의 Agent를 사용한다.

In [ ]:
from langgraph.prebuilt import create_react_agent

executor_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agent_executor = create_react_agent(
    executor_llm,
    tools,
)

### Replanner

지금까지의 실행 결과를 보고 두 가지 중 하나를 결정한다.

- 아직 할 일이 남았으면 -> 남은 계획을 수정해서 반환
- 충분하면 -> 최종 응답을 생성

In [ ]:
from typing import Union


class Response(BaseModel):
    """사용자에게 전달할 최종 응답"""
    response: str = Field(description="최종 응답")


class Act(BaseModel):
    """수행할 행동. Response 또는 Plan 중 하나를 반환한다."""
    action: Union[Response, Plan] = Field(
        description="수행할 행동. 최종 응답을 할 수 있으면 Response, 아직 단계가 남았으면 Plan을 반환."
    )


replanner_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """당신은 계획의 진행 상황을 평가하고 다음 행동을 결정하는 전문가입니다.

주어진 목표, 원래 계획, 그리고 지금까지 완료된 단계들을 보고:
1. 충분한 정보가 모였으면 최종 응답(Response)을 작성하세요.
2. 아직 부족하면 남은 단계를 수정/보완한 새 계획(Plan)을 반환하세요.

이미 완료된 단계를 다시 포함하지 마세요.""",
    ),
    (
        "human",
        """목표: {input}

원래 계획:
{plan}

완료된 단계:
{past_steps}

위 정보를 바탕으로 다음 행동을 결정하세요.""",
    ),
])

replanner_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
replanner = replanner_prompt | replanner_llm.with_structured_output(Act)

### 노드 함수 정의

위에서 만든 Chain/Agent를 State와 연결하는 노드 함수를 작성한다.

In [ ]:
from langchain_core.messages import HumanMessage


async def plan_step(state: PlanExecuteState):
    """계획 수립 노드"""
    result = await planner.ainvoke({"input": state["input"]})
    return {"plan": result.steps}


async def execute_step(state: PlanExecuteState):
    """현재 단계 실행 노드"""
    # 계획의 첫 번째 단계를 실행
    current_step = state["plan"][0]

    # 이전 결과를 컨텍스트로 포함
    past_context = ""
    if state.get("past_steps"):
        past_context = "\n\n지금까지의 결과:\n"
        for step, result in state["past_steps"]:
            past_context += f"- {step}: {result[:200]}\n"

    task = f"다음 작업을 수행하세요: {current_step}{past_context}"

    # ReAct Agent로 실행
    agent_response = await agent_executor.ainvoke(
        {"messages": [HumanMessage(content=task)]}
    )
    result = agent_response["messages"][-1].content

    return {"past_steps": [(current_step, result)]}


async def replan_step(state: PlanExecuteState):
    """계획 수정 노드"""
    result = await replanner.ainvoke({
        "input": state["input"],
        "plan": "\n".join(f"{i+1}. {s}" for i, s in enumerate(state["plan"])),
        "past_steps": "\n".join(
            f"- {step}: {result[:200]}" for step, result in state["past_steps"]
        ),
    })

    if isinstance(result.action, Response):
        return {"response": result.action.response}
    else:
        return {"plan": result.action.steps}

## 그래프 구성

전체 그래프의 흐름은 다음과 같다.

```
START → planner → executor → replanner → executor → replanner → ... → END
```

Replanner에서 최종 응답이 생성되면 END로 가고, 아직 계획이 남아있으면 다시 Executor로 돌아간다.

In [ ]:
from langgraph.graph import StateGraph, START, END


def should_continue(state: PlanExecuteState):
    """replanner 이후 분기: 응답이 있으면 종료, 없으면 계속 실행"""
    if state.get("response"):
        return "end"
    return "continue"


graph_builder = StateGraph(PlanExecuteState)

# 노드 추가
graph_builder.add_node("planner", plan_step)
graph_builder.add_node("executor", execute_step)
graph_builder.add_node("replanner", replan_step)

# 엣지 연결
graph_builder.add_edge(START, "planner")
graph_builder.add_edge("planner", "executor")
graph_builder.add_edge("executor", "replanner")
graph_builder.add_conditional_edges(
    "replanner",
    should_continue,
    {
        "continue": "executor",
        "end": END,
    },
)

graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

## 실행

멀티스텝이 필요한 질문으로 테스트한다.

In [ ]:
config = {"recursion_limit": 20}

inputs = {"input": "경쟁사 AI 서비스 가격을 조사해서 비교 보고서를 만들어줘. OpenAI, Google, Anthropic 3곳을 대상으로 해줘."}

async for event in graph.astream(inputs, config=config):
    for node_name, value in event.items():
        if node_name == "__end__":
            continue
        print(f"\n{'='*60}")
        print(f"[{node_name}]")
        print(f"{'='*60}")

        if node_name == "planner":
            print("계획:")
            for i, step in enumerate(value["plan"], 1):
                print(f"  {i}. {step}")

        elif node_name == "executor":
            step, result = value["past_steps"][-1]
            print(f"실행: {step}")
            print(f"결과: {result[:300]}...")

        elif node_name == "replanner":
            if value.get("response"):
                print("최종 응답 생성됨")
            elif value.get("plan"):
                print("수정된 계획:")
                for i, step in enumerate(value["plan"], 1):
                    print(f"  {i}. {step}")

In [ ]:
# 최종 응답 확인
final_state = await graph.ainvoke(inputs, config=config)
print(final_state["response"])

## Re-planning

Re-planning이 어떻게 동작하는지 좀 더 자세히 살펴보자.

Replanner는 매 단계 실행 후 호출되며, 다음 중 하나를 결정한다.

| 상황 | Replanner의 판단 | 결과 |
|------|------------------|------|
| 검색 결과가 충분함 | 남은 계획을 줄이거나 최종 응답 생성 | 효율적 |
| 검색 결과가 부족함 | 추가 검색 단계를 계획에 삽입 | 품질 향상 |
| 예상과 다른 결과 | 계획 자체를 수정 | 유연한 대응 |
| 모든 단계 완료 | 최종 응답 생성 | 종료 |

이것이 Plan-and-Execute의 핵심 강점이다. 단순히 계획을 세우고 실행하는 것이 아니라, **중간 결과에 따라 계획을 수정**할 수 있다.

Re-planning 시 주의할 점:
- `past_steps`에 이전 결과가 모두 누적되어 있으므로, Replanner가 맥락을 파악할 수 있다
- 무한 루프를 방지하기 위해 `recursion_limit`을 설정한다
- `past_steps`가 너무 길어지면 컨텍스트 윈도우를 초과할 수 있으므로, 결과를 요약하는 전략도 고려해야 한다

In [ ]:
# Re-planning이 잘 보이는 예시: 모호한 질문으로 시작
inputs2 = {"input": "최근 AI 트렌드를 분석해서 요약해줘"}

async for event in graph.astream(inputs2, config=config):
    for node_name, value in event.items():
        if node_name == "__end__":
            continue
        print(f"\n--- [{node_name}] ---")

        if node_name == "planner":
            for i, step in enumerate(value["plan"], 1):
                print(f"  {i}. {step}")

        elif node_name == "executor":
            step, result = value["past_steps"][-1]
            print(f"  실행: {step}")
            print(f"  결과 길이: {len(result)}자")

        elif node_name == "replanner":
            if value.get("response"):
                print("  -> 최종 응답 생성")
                print(f"  응답 미리보기: {value['response'][:200]}...")
            elif value.get("plan"):
                print("  -> 계획 수정됨")
                for i, step in enumerate(value["plan"], 1):
                    print(f"     {i}. {step}")

## 컨텍스트 관리

멀티스텝 Agent에서는 Tool 호출 결과 + 검색 결과 + 대화 히스토리가 동시에 쌓이면서 context window 압박이 생긴다.

### 관리 전략

| 전략 | 방법 | 적용 시점 |
|------|------|----------|
| **결과 요약** | Tool 결과를 LLM으로 요약하여 저장 | Tool 결과가 길 때 |
| **Truncation** | 최대 길이를 정해 자르기 | 항상 (안전장치) |
| **우선순위 관리** | 최신 결과 > 오래된 결과, 현재 단계와 관련된 결과 우선 | past_steps가 많을 때 |
| **선택적 전달** | 현재 단계에 필요한 정보만 프롬프트에 포함 | 단계가 독립적일 때 |

In [ ]:
# 컨텍스트 관리 예시: Tool 결과 요약
def summarize_step_result(result: str, max_length: int = 500) -> str:
    """긴 Tool 결과를 요약한다."""
    if len(result) <= max_length:
        return result
    
    summary = llm.invoke(
        f"다음 내용을 핵심만 {max_length}자 이내로 요약해:\n\n{result}"
    )
    return summary.content


# 컨텍스트 우선순위: 최근 N개 결과만 프롬프트에 포함
def get_relevant_context(past_steps: list[tuple], max_steps: int = 3) -> str:
    """최근 N개 단계의 결과만 반환한다."""
    recent = past_steps[-max_steps:]
    return "\n\n".join(
        f"[{step}] {result[:200]}" for step, result in recent
    )

실무에서는 이 전략들을 조합한다. 예를 들어, 각 단계 결과를 요약하여 저장하고(`summarize_step_result`), 프롬프트에는 최근 3개 단계만 포함하는(`get_relevant_context`) 방식이다.

---

## 셀프 체크리스트

- [ ] Plan-and-Execute와 ReAct의 차이를 설명할 수 있다
- [ ] `plan`, `past_steps`, `response`의 역할과 reducer 설정 이유를 안다
- [ ] Planner, Executor, Replanner 각 노드의 역할을 이해한다
- [ ] Structured Output으로 계획을 생성하는 방법을 안다
- [ ] Re-planning이 필요한 상황과 동작 방식을 설명할 수 있다
- [ ] `recursion_limit`으로 무한 루프를 방지하는 방법을 안다